<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

a page is worth reviewing fo refresh if it hasn't been touched in over 90days  and still gets a meaningful amount of search traffic meaning there's still an audience to win back.Among pages meeting both conditions, prioritize by how much traffic they get, since a fix there has more upside.

In [2]:
import pandas as pd

# Load the dataset directly from the provided URL
df = pd.read_csv('https://raw.githubusercontent.com/ritakimani9-lang/machinelearning/main/data/raw/content_refresh_anonymized.csv')

# Validation only — is_declining_label is NEVER a feature in the score below.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

bins = [-1, 14, 30, 90, 180, 10000]
labels = ['0-14', '15-30', '31-90', '91-180', '181+']
df['fresh_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)

t1 = df.groupby('fresh_bucket', observed=True)['is_declining_label'].agg(['mean', 'count'])
t1['mean'] = (t1['mean'] * 100).round(1)
print(t1)

              mean  count
fresh_bucket             
0-14          51.0   3933
15-30         51.2  16547
31-90         58.9    175
91-180        61.1   9171
181+          47.1    174


Verdict: CONFIRMED (directionally). The two well-powered buckets (n=20,480 combined fresh vs n=9,171 stale) show a clean, meaningful gap: 51% decline when fresh vs 61% when stale — a real ~10-point difference on strong sample sizes. The two middle/tail buckets (n=175 and n=174) don't fit the trend, but they're too small to trust — flag them as noise, don't let them override the two solid buckets.

In [3]:
#signal 2 search volume (flag linked behind quick win logic)
bins = [-1, 0, 50, 500, 5000, 1_000_000]
labels = ['0', '1-50', '51-500', '501-5000', '5000+']
df['volume_bucket'] = pd.cut(df['search_volume'], bins=bins, labels=labels)

t2 = df.groupby('volume_bucket', observed=True)['is_declining_label'].agg(['mean', 'count'])
t2['mean'] = (t2['mean'] * 100).round(1)
print(t2)

               mean  count
volume_bucket             
0              63.2  11081
1-50           52.6  12300
51-500         50.9   3123
501-5000       43.5    876
5000+          50.7    152


Verdict: OPPOSITE. My hypothesis going in was "pages targeting higher-volume keywords are more exposed/competitive, so they'd decline more." The data says the reverse: decline rate falls steadily as volume rises (63% → 44%), except the top bucket, which is small (n=152) and unreliable. This is a useful negative result — it means I should not treat high search volume as a decline predictor. I still use impressions_90d in the score below, but only as an opportunity-size multiplier (bigger audience = bigger payoff if fixed), never as evidence a page is declining — that's an important distinction to be explicit about, since it's the opposite of what I originally assumed


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- The rule (uses only pre-decision signals; is_declining_label is not touched here) ---
stale = df['freshness_tier'].isin(['91-180', '181+']).astype(int)
visible = df['impression_tier'].isin(['moderate', 'good', 'excellent']).astype(int)

df['score'] = stale * visible * df['impressions_90d']
df['reason_code'] = df['score'].apply(lambda s: 'STALE_BUT_VISIBLE' if s > 0 else 'NOT_FLAGGED')
df['action'] = df['score'].apply(lambda s: 'REVIEW_FOR_REFRESH' if s > 0 else 'NO_ACTION')

ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
print(f"Flagged: {(ranked['score'] > 0).sum()} of {len(ranked)} rows")

import os
os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

Flagged: 7234 of 30000 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = ranked.head(10)

for i, row in top10.iterrows():
    print(f"Rank {i+1}")
    print(f"  action: {row['action']} | reason: {row['reason_code']}")
    print(f"  days_since_update: {row['days_since_last_update']}, impression_tier: {row['impression_tier']}")
    print(f"  impressions_90d: {row['impressions_90d']}, avg_position: {row['avg_position']}, ctr: {row['ctr']}")
    print()

Rank 1
  action: REVIEW_FOR_REFRESH | reason: STALE_BUT_VISIBLE
  days_since_update: 104, impression_tier: excellent
  impressions_90d: 517715, avg_position: 4.2, ctr: 0.14

Rank 2
  action: REVIEW_FOR_REFRESH | reason: STALE_BUT_VISIBLE
  days_since_update: 104, impression_tier: excellent
  impressions_90d: 443434, avg_position: 27.9, ctr: 0.21

Rank 3
  action: REVIEW_FOR_REFRESH | reason: STALE_BUT_VISIBLE
  days_since_update: 104, impression_tier: excellent
  impressions_90d: 347399, avg_position: 4.2, ctr: 0.53

Rank 4
  action: REVIEW_FOR_REFRESH | reason: STALE_BUT_VISIBLE
  days_since_update: 104, impression_tier: excellent
  impressions_90d: 309910, avg_position: 5.6, ctr: 0.16

Rank 5
  action: REVIEW_FOR_REFRESH | reason: STALE_BUT_VISIBLE
  days_since_update: 104, impression_tier: excellent
  impressions_90d: 309192, avg_position: 2.0, ctr: 0.87

Rank 6
  action: REVIEW_FOR_REFRESH | reason: STALE_BUT_VISIBLE
  days_since_update: 104, impression_tier: excellent
  impression

REVIEW_FOR_REFRESH, High. Ranks well (4.2) but converts at only 0.14% and is genuinely declining (−44.8%) — a real relevance or snippet problem, not just noise. Wrong if: the low CTR turns out to be a SERP-feature issue (e.g. a featured snippet above it) rather than a content problem — a refresh wouldn't fix that.
REVIEW_FOR_REFRESH, Low. Flagged purely on size (443K impressions) and staleness — but it's actually stable (+1.4%), not declining. This is where the rule's core assumption breaks. Wrong if: — it already is wrong; this page doesn't need urgent attention, it's here because it's big, not because it's hurting.
REVIEW_FOR_REFRESH, High. Good position (4.2), genuinely declining (−36.5%), commercial intent (money page). Solid pick. Wrong if: the decline is seasonal/temporary rather than structural.
REVIEW_FOR_REFRESH, High. Transactional (high business value), declining (−41.8%), still ranks well. Strong candidate. Wrong if: the drop is due to a competitor's temporary promotion rather than this page's own quality.
REVIEW_FOR_REFRESH, Medium. Already ranks top_3 (2.0!) yet is declining (−37.3%) and its CTR (0.87%) is below what top_3 pages typically pull. Interesting — a page losing traffic despite great position is a real signal. Wrong if: the underperformance is really a SERP layout change (ads/snippets crowding it out) rather than the content itself.
REVIEW_FOR_REFRESH, Low. Flagged for size and staleness only — actually stable (+0.5%), and CTR (0.05%) is just low for its rank. Wrong if: — already weak; nothing here suggests this page has a problem beyond being unglamorous.
REVIEW_FOR_REFRESH, Medium. Trend is −17.2% but labeled "stable" only because it's just short of the −20% cutoff for "down" — a real borderline case the label's binary cutoff hides. Wrong if: you treat "stable" at face value and skip it — the underlying number says otherwise.
REVIEW_FOR_REFRESH, High. Weak position (26.2), weak CTR (0.06%), and genuinely declining (−33.8%) — three signals agreeing. One of the cleaner true positives. Wrong if: word_count/content quality checks out fine and the real cause is external (e.g. a manual/algorithmic penalty), which a content refresh alone won't fix.
REVIEW_FOR_REFRESH, Low. Similar borderline case to #7 (−11.6%, labeled stable) — flagged mainly on size. Wrong if: you expect this to behave like the confirmed decliners above it; it's a weaker case than most of the list.
REVIEW_FOR_REFRESH, High. 208,678 impressions and literally zero clicks — that's a striking anomaly on its own, plus a confirmed steep decline (−43.4%). Likely the strongest true positive in the list. Wrong if: the 0.00% CTR is a tracking/rounding artifact rather than real — worth verifying the raw click count isn't just a very small nonzero number rounded down.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick, honestly stated: I ran precision@K against is_declining_label (used only for evaluation, never as an input):

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['is_declining_label'].mean()
for k in [10, 20, 50, 100]:
    print(f"precision@{k}: {precision_at_k(df['score'], df['is_declining_label'], k):.2f}  (base rate: {base_rate:.2f})")

precision@10: 0.60  (base rate: 0.54)
precision@20: 0.45  (base rate: 0.54)
precision@50: 0.44  (base rate: 0.54)
precision@100: 0.38  (base rate: 0.54)


my score, dominated by raw impressions among the stale+visible pool, is really an impact-size ranker, not a good decline-predictor. That's a legitimate weak spot to name — it's exactly what next week's model needs to beat.

Leakage check: trend_direction, trend_pct, and is_declining_label appear only in the validation/evaluation code (Sections 1 and 4), never inside the score formula in Section 2. content_id/client_id are used for display only, not as features. No future-window columns (*_last30 used only descriptively, not combined with a label from the same window) went into the score. This should be clean.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.